# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [20]:
# # Only needed for Udacity workspace

# import importlib.util
# import sys

# # Check if 'pysqlite3' is available before importing
# if importlib.util.find_spec("pysqlite3") is not None:
#     import pysqlite3
#     sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [21]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [22]:
load_dotenv()

assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None
assert os.getenv("CHROMA_API_KEY") is not None
assert os.getenv("CHROMA_TENANT") is not None
assert os.getenv("CHROMA_DATABASE") is not None

OPENAI_BASE_URL = os.getenv(
    "OPENAI_BASE_URL",
    "https://openai.vocareum.com/v1"
)

### VectorDB Instance

In [23]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# chroma_client = chromadb.PersistentClient(path="chromadb")

# chroma_client = chromadb.CloudClient(
#   api_key=os.getenv("CHROMA_API_KEY"),
#   tenant=os.getenv("CHROMA_TENANT"),
#   database=os.getenv("CHROMA_DATABASE")
# )

chroma_client = chromadb.PersistentClient(
    path="chromadb"
)



### Collection

In [24]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [25]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.create_collection(
   name="udaplay",
   embedding_function=embedding_fn,
   get_or_create=True
)

### Add documents

In [26]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    games = data if isinstance(data, list) else [data]

    ids = []
    documents = []
    metadatas = []

    file_id = os.path.splitext(file_name)[0]

    for index, game in enumerate(games):

        content = (
            f"[{game['Platform']}] "
            f"{game['Name']} "
            f"({game['YearOfRelease']}) - "
            f"{game['Description']}"
        )

        doc_id = f"{file_id}-{index}"

        metadata = {}

        for key, value in game.items():
            if value is None:
                continue

            if isinstance(value, (str, int, float, bool)):
                metadata[key] = value
            else:
                metadata[key] = str(value)

        ids.append(doc_id)
        documents.append(content)
        metadatas.append(metadata)

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )

    print(f"{file_name}: {len(games)} documents added")

    items_in_collection = collection.count()
    assert items_in_collection > 0 
    print(f"Items in collection: {items_in_collection}")

001.json: 1 documents added
Items in collection: 1
002.json: 1 documents added
Items in collection: 2
003.json: 1 documents added
Items in collection: 3
004.json: 1 documents added
Items in collection: 4
005.json: 1 documents added
Items in collection: 5
006.json: 1 documents added
Items in collection: 6
007.json: 1 documents added
Items in collection: 7
008.json: 1 documents added
Items in collection: 8
009.json: 1 documents added
Items in collection: 9
010.json: 1 documents added
Items in collection: 10
011.json: 1 documents added
Items in collection: 11
012.json: 1 documents added
Items in collection: 12
013.json: 1 documents added
Items in collection: 13
014.json: 1 documents added
Items in collection: 14
015.json: 1 documents added
Items in collection: 15
video_games.json: 198 documents added
Items in collection: 213


In [27]:
question = "¿Which year was launched Batman: Arkham City for X360?"

results = collection.query(
    query_texts=[question],
    n_results=3,
    include=['documents', 'metadatas']
)

print(results)

{'ids': [['video_games-197', 'video_games-145', '004-0']], 'embeddings': None, 'documents': [['[X360] Batman: Arkham City (2011) - An action-adventure game set in a massive urban prison where Batman faces villains including the Joker and Hugo Strange. It combines combat, stealth, exploration and puzzle solving.', '[PS3] Batman: Arkham City (2011) - An action-adventure game set in a massive urban prison where Batman faces villains including the Joker and Hugo Strange. It combines combat, stealth, exploration and puzzle solving.', "[PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains."]], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [[{'Genre': 'Action', 'Publisher': 'Warner Bros. Interactive Entertainment', 'Name': 'Batman: Arkham City', 'Platform': 'X360', 'YearOfRelease': 2011, 'Description': 'An action-adventure game set in a massive urban pri